In [ ]:
pip install torch transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Step 1: Setup started...")
DEVICE = torch.device("cpu")
MODEL_NAME = "mental/mental-bert-base-uncased"
HF_TOKEN = "YOUR_TOKEN_HERE" # <--- PASTE TOKEN HERE

print("Step 2: Downloading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

print("Step 3: Downloading Base Model (This is the heavy part!)...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6, token=HF_TOKEN)

print("Step 4: Base model secured! Locating your custom weights...")
MODEL_PATH = r"C:\Users\KBhagyaRekha\Desktop\Cognisafe V2\Minor\best_model.pt"
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)

print("Step 5: Injecting your weights into the brain...")
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()
print("Step 6: ✅ SUCCESS!")

In [ ]:
# The labels in alphabetical order (same as your training script)
labels = ['Anxiety', 'BPD', 'bipolar', 'depression', 'mentalillness', 'schizophrenia']

def predict_mental_health(text):
    # 1. Prep the text for the AI
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding="max_length")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    # 2. Get the AI's prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Calculate percentages (confidence) for each class
        probabilities = torch.nn.functional.softmax(logits, dim=1).squeeze()
        
    # 3. Find the winner
    confidence = torch.max(probabilities).item() * 100
    predicted_id = torch.argmax(probabilities).item()
    predicted_label = labels[predicted_id]
    
    print(f"\nText: '{text}'")
    print(f"Prediction: {predicted_label} (Confidence: {confidence:.2f}%)")
    print("-" * 40)

# The Ultimate Test
predict_mental_health("I'm so exhausted all the time and I can't find the energy to get out of bed. Nothing brings me joy anymore.")
predict_mental_health("My heart is constantly racing, my palms are sweating, and I feel like everyone is judging me.")